In [ ]:
"""
Patient Experience Analysis — MedCore
=====================================
Analyzes 3 years of patient feedback: what patients are unhappy about,
where it happens, and what it costs the organization.

Structure:
  1. Setup
  2. Data loading & cleaning
  3. Exploratory analysis
  4. Booking delay & no-show analysis
  5. Cost of no-shows
  6. Conclusions
"""

# ============================================================
# 1. SETUP
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

# coral = negative / at-risk, teal = positive, gold = neutral
COLOR_NEG, COLOR_POS, COLOR_MID = "#D8492F", "#0F5C5C", "#B8862F"


# ============================================================
# 2. DATA LOADING & CLEANING
# ============================================================
df = pd.read_csv("patient_experience_master.csv")

# Keep valid ratings (1-5) and rows with a usable no-show flag
df = df[df["rating"].isin([1, 2, 3, 4, 5])].copy()
df = df.dropna(subset=["future_no_show_flag"])
df["future_no_show_flag"] = df["future_no_show_flag"].astype(int)

# Fill missing booking delays with the median, parse dates
df["wait_days"] = df["wait_days"].fillna(df["wait_days"].median())
df["feedback_date"] = pd.to_datetime(df["feedback_date"], errors="coerce")

# Rows without a complaint_category are happy patients (all rated 4-5)
df["theme"] = df["complaint_category"].fillna("Positive / No Complaint")

overall = df["at_risk"].mean()
print(f"{len(df):,} responses | overall At-Risk rate: {overall:.1%}")


# ============================================================
# 3. EXPLORATORY ANALYSIS
# ============================================================

# --- 3.1 Rating distribution ---
# ~38.6% of responses are At-Risk: a widespread issue, not a small group
counts = df["rating"].value_counts().sort_index()
colors = [COLOR_NEG if r <= 3 else COLOR_POS for r in counts.index]
counts.plot(kind="bar", color=colors, figsize=(8, 4))
plt.title(f"Rating distribution — {overall:.1%} of responses are At Risk (coral)")
plt.xlabel("Rating"); plt.ylabel("Responses"); plt.xticks(rotation=0)
plt.show()

# --- 3.2 At-Risk rate by theme ---
# Billing / Provider / Facilities complaints are At-Risk 100% of the time
themes = df.groupby("theme").agg(count=("theme", "size"), risk=("at_risk", "mean"))
neg = themes[themes["risk"] > 0].sort_values("risk")
(neg["risk"] * 100).plot(kind="barh", color=COLOR_NEG, figsize=(8, 4))
plt.axvline(overall * 100, color="gray", linestyle="--", label="org average")
plt.title("Share of each complaint theme classified At Risk")
plt.xlabel("At-Risk rate (%)"); plt.legend()
plt.show()

# --- 3.3 Volume vs severity ---
# Wait Time is the most common complaint but not the most damaging
plt.figure(figsize=(8, 5))
plt.scatter(neg["count"], neg["risk"] * 100, s=neg["count"] / 50,
            color=COLOR_NEG, alpha=0.7)
for name, row in neg.iterrows():
    plt.annotate(name, (row["count"], row["risk"] * 100 + 2), ha="center")
plt.title("Volume vs severity — frequent complaints are not the most damaging")
plt.xlabel("Number of complaints"); plt.ylabel("At-Risk rate (%)")
plt.show()

# --- 3.4 Department hotspots ---
# At-Risk rates range ~20.5% to ~57.9% across departments (~3x difference)
dept = df.groupby("department_id").agg(count=("rating", "size"), risk=("at_risk", "mean"))
dept = dept[dept["count"] >= 50].sort_values("risk", ascending=False)
(dept["risk"] * 100).plot(kind="bar", color=COLOR_POS, figsize=(12, 4))
plt.axhline(overall * 100, color="gray", linestyle="--", label="org average")
plt.title(f"At-Risk rate by department ({dept['risk'].min():.1%} to {dept['risk'].max():.1%})")
plt.ylabel("At-Risk rate (%)"); plt.xticks(fontsize=6); plt.legend()
plt.show()

# --- 3.5 Quarterly trend ---
# Flat at ~38-39% for 3 years: a structural problem, not a recent spike
dated = df.dropna(subset=["feedback_date"])
trend = dated.groupby(dated["feedback_date"].dt.to_period("Q"))["at_risk"].mean()
(trend * 100).plot(marker="o", color=COLOR_NEG, figsize=(9, 4))
plt.ylim(30, 45)
plt.title("Quarterly At-Risk rate — flat for 3 years (structural, not a spike)")
plt.ylabel("At-Risk rate (%)")
plt.show()


# ============================================================
# 4. BOOKING DELAY & NO-SHOW ANALYSIS
# ============================================================

# --- 4.1 Booking delay vs At-Risk ---
# Dissatisfaction climbs with scheduling lag (~20% within 3 days vs ~54% past 30)
df["wait_bucket"] = pd.cut(df["wait_days"], [0, 3, 7, 14, 21, 30, 70],
                           labels=["0-3", "4-7", "8-14", "15-21", "22-30", "31+"])
wait = df.groupby("wait_bucket", observed=True)["at_risk"].mean()
(wait * 100).plot(kind="bar", color=COLOR_MID, figsize=(8, 4))
plt.axhline(overall * 100, color="gray", linestyle="--", label="org average")
plt.title("At-Risk rate rises with booking delay")
plt.xlabel("Days between scheduling and appointment"); plt.ylabel("At-Risk rate (%)")
plt.xticks(rotation=0); plt.legend()
plt.show()

# --- 4.2 Booking delay by rating level ---
# Rating-1 patients waited ~37 days on average vs ~16 days for rating-5
delay = df.groupby("rating")["wait_days"].mean()
colors = [COLOR_NEG if r <= 3 else COLOR_POS for r in delay.index]
delay.plot(kind="bar", color=colors, figsize=(8, 4))
plt.title("Average booking delay by rating — unhappy patients waited 2x longer")
plt.xlabel("Patient rating"); plt.ylabel("Avg days from scheduling to appointment")
plt.xticks(rotation=0)
plt.show()

# --- 4.3 Future no-shows ---
# At-Risk patients no-show ~6x more often — the cost of doing nothing
noshow = df.groupby("at_risk")["future_no_show_flag"].mean()
(noshow * 100).rename({False: "Satisfied", True: "At Risk"}).plot(
    kind="bar", color=[COLOR_POS, COLOR_NEG], figsize=(6, 4))
plt.title(f"At-Risk patients no-show {noshow[True]/noshow[False]:.1f}x more often")
plt.ylabel("Future no-show rate (%)"); plt.xticks(rotation=0)
plt.show()


# ============================================================
# 5. COST OF NO-SHOWS
# ============================================================
# The average completed visit generates ~$617 in claims;
# every no-show forfeits that revenue.
avg_revenue = df.loc[df["has_claim"] & (df["total_claim_amount"] > 0),
                     "total_claim_amount"].mean()
n_noshows = (df["status"] == "No Show").sum()
total_cost = n_noshows * avg_revenue

print(f"Average revenue per visit:  ${avg_revenue:,.0f}")
print(f"No-shows in dataset:        {n_noshows:,}")
print(f"Estimated lost revenue:     ${total_cost/1e6:.1f}M over 3 years "
      f"(~${total_cost/3/1e6:.1f}M per year)")

pd.Series({"Revenue kept\n(completed visits)": (df["status"] == "Completed").sum() * avg_revenue / 1e6,
           "Revenue lost\n(no-shows)": total_cost / 1e6}).plot(
    kind="bar", color=[COLOR_POS, COLOR_NEG], figsize=(6, 4))
plt.title(f"No-shows cost MedCore ≈ ${total_cost/1e6:.0f}M in lost visit revenue")
plt.ylabel("Estimated revenue ($M)"); plt.xticks(rotation=0)
plt.show()

# --- 5.1 Cost attributable to At-Risk patients ---
# At-Risk patients account for ~79.5% of all no-shows (~$15.9M over 3 years)
ns = df[df["status"] == "No Show"].groupby("at_risk").size()
share_ar = ns[True] / ns.sum()
cost_ar = ns[True] * avg_revenue

print(f"No-shows — Satisfied: {ns[False]:,} | At-Risk: {ns[True]:,} "
      f"({share_ar:.1%} of all no-shows)")
print(f"Cost of At-Risk no-shows: ${cost_ar/1e6:.1f}M over 3 years "
      f"(~${cost_ar/3/1e6:.1f}M per year)")

# Excess no-shows: even satisfied patients no-show ~3.8% of the time, so we
# only count the difference — the cost of dissatisfaction itself (~$4.4M/yr)
rates = df.groupby("at_risk")["future_no_show_flag"].mean()
excess = (rates[True] - rates[False]) * df["at_risk"].sum()
excess_cost = excess * avg_revenue
print(f"Future no-show rate — Satisfied: {rates[False]:.1%} | At-Risk: {rates[True]:.1%}")
print(f"Excess no-shows attributable to dissatisfaction: {excess:,.0f} "
      f"= ${excess_cost/1e6:.1f}M over 3 years (~${excess_cost/3/1e6:.1f}M per year)")

pd.Series({"Satisfied\npatients": ns[False] * avg_revenue / 1e6,
           "At-Risk\npatients": cost_ar / 1e6}).plot(
    kind="bar", color=[COLOR_POS, COLOR_NEG], figsize=(6, 4))
plt.title(f"At-Risk patients drive {share_ar:.0%} of no-show losses "
          f"(≈${cost_ar/1e6:.0f}M)")
plt.ylabel("Lost visit revenue ($M)"); plt.xticks(rotation=0)
plt.show()


# ============================================================
# 6. CONCLUSIONS
# ============================================================
# Dissatisfaction is widespread (~38.6% At-Risk) and stable for 3 years —
#   a structural problem, not a recent spike.
# It is concentrated: some departments run ~3x the At-Risk rate of others,
#   and Billing / Provider / Facilities complaints are always At-Risk.
# It is driven partly by access: longer booking delays go hand in hand
#   with lower ratings and higher At-Risk rates.
# And it is expensive: At-Risk patients drive ~80% of no-show losses,
#   with excess no-shows costing ~$4.4M per year.
# More in-depth conclusions included in our analysis document. 